In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, col, to_timestamp, current_timestamp, lit, count, when
import src.bronze as bronze
import src.silver as silver



In [0]:
stop_times = spark.sql('select * from warsaw_transport_bronze_stop_times')
stop_times.show(5)

In [0]:
stop_time = silver.select_stop_time(stop_times)
stop_time.show()

In [0]:
from pyspark.sql.functions import expr

to_timestamp_df = stop_times.select(
    expr("try_to_timestamp(arrival_time, 'HH:mm:ss')").alias('arrival_time_ts')
)
to_timestamp_df.show()


In [0]:
from pyspark.sql.functions import expr

departure_time_df = stop_times.select(expr("try_to_timestamp(departure_time, 'HH:mm:ss')").alias('departure_time_st'))
departure_time_df.show()

In [0]:
stop_times = stop_times.withColumn('stop_id', col('stop_id').cast('string'))

In [0]:
spark.sql('create database if not exists workspace_warsaw_transport_silver')

stop_times = stop_times \
    .withColumn('ingest_time', current_timestamp()) \
    .withColumn('ingest_source', lit("ZTM_Warsaw_GTFS"))

stop_times.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('workspace_warsaw_transport_silver.stop_times') 

In [0]:
stops = spark.table('warsaw_transport_bronze_stops')
stops.show(5)
stops.printSchema()

In [0]:
stops = stops \
    .withColumn('stop_id', col('stop_id').cast('string')) \
    .withColumn('stop_code', col('stop_code').cast('string'))

In [0]:
stops = stops.withColumn('ingest_time', current_timestamp()) \
    .withColumn('ingest_source', lit('ZTM_Warsaw_GTFS'))

stops.write.format('delta') \
    .mode('overwrite') \
        .option('overwriteSchema', 'true') \
            .saveAsTable('workspace_warsaw_transport_silver.stops')

In [0]:
trip = spark.table('workspace.default.warsaw_transport_bronze_trips')
trip.show()
trip.printSchema()

In [0]:
null_cols = trip.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in trip.columns
])
null_cols.show()

In [0]:
trip = trip.withColumn('route_id', col('route_id').cast('string'))

In [0]:
trip = trip \
    .withColumn('ingest_time', current_timestamp()) \
    .withColumn('ingest_source', lit('ZTM_Warsaw_GTFS'))

trip.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('workspace_warsaw_transport_silver.trip') 

In [0]:
routes = spark.table('warsaw_transport_bronze_routes')
routes.show(5)
routes.printSchema()

In [0]:
routes = routes \
.withColumn('agency_id', col('agency_id').cast('string')) \
    .withColumn('route_id', col('route_id').cast('string')) 

In [0]:
routes = routes.withColumn('ingest_time', current_timestamp()) \
    .withColumn('ingest_source', lit('ZTM_Warsaw_GTFS'))

routes.write.format('delta') \
    .mode('overwrite') \
        .option('overwriteSchema', 'true') \
            .saveAsTable('workspace_warsaw_transport_silver.routes')    